In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/cleaned/master_dataset.csv')

conn = sqlite3.connect('../sql/ecommerce.db')
df.to_sql('orders_master', conn, if_exists='replace', index=False)

print("Loaded into SQLite:", df.shape)

Loaded into SQLite: (117891, 35)


In [2]:
query = "SELECT * FROM orders_master LIMIT 5"
pd.read_sql(query, conn)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_city,customer_state,payment_sequential,payment_type,payment_installments,payment_value,review_score,order_year_month,delivery_delay_days,total_item_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,sao paulo,SP,1.0,credit_card,1.0,18.12,4.0,2017-10,-8.0,38.71
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,sao paulo,SP,3.0,voucher,1.0,2.00,4.0,2017-10,-8.0,38.71
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,87285b34884572647811a353c7ac498a,...,sao paulo,SP,2.0,voucher,1.0,18.59,4.0,2017-10,-8.0,38.71
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,...,barreiras,BA,1.0,boleto,1.0,141.46,4.0,2018-07,-6.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,aa4383b373c6aca5d8797843e5594415,...,vianopolis,GO,1.0,credit_card,3.0,179.12,5.0,2018-08,-18.0,179.12


In [3]:
query = """
SELECT 
    order_year_month,
    SUM(total_item_value) AS monthly_revenue,
    SUM(SUM(total_item_value)) OVER (ORDER BY order_year_month) AS running_total
FROM orders_master
GROUP BY order_year_month
ORDER BY order_year_month;
"""
monthly_revenue = pd.read_sql(query, conn)
monthly_revenue

,order_year_month,monthly_revenue,running_total
0,2016-09,354.75,354.75
1,2016-10,58550.03,58904.78
2,2016-12,19.62,58924.40
3,2017-01,147144.50,206068.90
4,2017-02,302466.08,508534.98
5,2017-03,458909.64,967444.62
6,2017-04,449077.80,1416522.42
7,2017-05,631757.66,2048280.08
8,2017-06,528082.58,2576362.66
9,2017-07,629462.83,3205825.49


In [4]:
query = """
SELECT 
    product_category_name_english,
    COUNT(DISTINCT order_id) AS num_orders,
    SUM(total_item_value) AS total_revenue
FROM orders_master
GROUP BY product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,product_category_name_english,num_orders,total_revenue
0,health_beauty,8836,1487194.54
1,watches_gifts,5624,1358593.59
2,bed_bath_table,9417,1317200.10
3,sports_leisure,7720,1199992.52
4,computers_accessories,6689,1099946.35
5,furniture_decor,6449,951935.44
6,housewares,5884,821298.03
7,cool_stuff,3632,751210.41
8,auto,3897,712725.47
9,garden_tools,3518,624485.83


In [5]:
query = """
SELECT 
    customer_unique_id,
    MAX(order_purchase_timestamp) AS last_purchase_date,
    COUNT(DISTINCT order_id) AS frequency,
    SUM(total_item_value) AS monetary
FROM orders_master
GROUP BY customer_unique_id;
"""
rfm_raw = pd.read_sql(query, conn)
rfm_raw.head()

,customer_unique_id,last_purchase_date,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,1,27.19
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,1,196.89


In [6]:
rfm_raw['last_purchase_date'] = pd.to_datetime(rfm_raw['last_purchase_date'])
snapshot_date = rfm_raw['last_purchase_date'].max() + pd.Timedelta(days=1)

rfm_raw['recency'] = (snapshot_date - rfm_raw['last_purchase_date']).dt.days

rfm = rfm_raw[['customer_unique_id', 'recency', 'frequency', 'monetary']]
rfm.head()

,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,116,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,119,1,27.19
2,0000f46a3911fa3c0805444483337064,542,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,326,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,293,1,196.89


In [7]:
rfm.to_csv('../data/cleaned/rfm_table.csv', index=False)
print("RFM table saved:", rfm.shape)

RFM table saved: (95420, 4)


In [8]:
query = """
SELECT 
    customer_unique_id,
    MIN(order_year_month) AS cohort_month,
    order_year_month AS order_month
FROM orders_master
GROUP BY customer_unique_id, order_year_month;
"""
cohort_data = pd.read_sql(query, conn)
cohort_data.head()

,customer_unique_id,cohort_month,order_month
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05,2018-05
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05,2018-05
2,0000f46a3911fa3c0805444483337064,2017-03,2017-03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10,2017-10
4,0004aac84e0df4da2b147fca70cf8255,2017-11,2017-11


In [9]:
cohort_data['cohort_month'] = pd.to_datetime(cohort_data['cohort_month'])
cohort_data['order_month'] = pd.to_datetime(cohort_data['order_month'])

cohort_data['period_number'] = (
    (cohort_data['order_month'].dt.year - cohort_data['cohort_month'].dt.year) * 12 +
    (cohort_data['order_month'].dt.month - cohort_data['cohort_month'].dt.month)
)

cohort_counts = cohort_data.groupby(['cohort_month', 'period_number'])['customer_unique_id'].nunique().reset_index()
cohort_pivot = cohort_counts.pivot(index='cohort_month', columns='period_number', values='customer_unique_id')

cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0).round(3)

retention.head(10)

period_number,0
cohort_month,
2016-09-01,1.0
2016-10-01,1.0
2016-12-01,1.0
2017-01-01,1.0
2017-02-01,1.0
2017-03-01,1.0
2017-04-01,1.0
2017-05-01,1.0
2017-06-01,1.0


In [10]:
query = """
SELECT 
    seller_id,
    COUNT(DISTINCT order_id) AS num_orders,
    SUM(total_item_value) AS total_revenue
FROM orders_master
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,seller_id,num_orders,total_revenue
0,53243585a1d6dc2643021fd1853d8905,358,258882.28
1,4869f7a5dfa277a7dca6462dcf3b52b2,1132,258625.52
2,7c67e1448b00f6e969d365cea6b010ab,982,253711.53
3,4a3ca9315b744ce9f8e9374361493884,1806,252310.03
4,fa1c13f2614d7b5c4749cbc52fecda94,585,214454.82
5,da8622b14eb17ae2831f4ac5b9dab84a,1314,196467.95
6,7e93a43ef30c4f03f38b393420bc753a,336,189475.90
7,1025f0e2d44d7041d6cf58b6550e0bfa,915,178012.02
8,7a67c85e85bb2ce8582c35f2203ad736,1160,172500.41
9,955fee9216a65b617aa5c0531780ce60,1287,163183.53


In [12]:
print(retention.columns.tolist())

[0]


In [14]:
if retention.shape[1] > 1:
    month1_retention = retention.iloc[:, 1].mean() * 100
    print(f"Month-1 retention: {month1_retention:.1f}%")
else:
    month1_retention = 0
    print("No customers had a repeat purchase in a later calendar month — retention table only has period 0.")

No customers had a repeat purchase in a later calendar month — retention table only has period 0.


In [16]:
# Recency
recency_pct = (rfm['recency'] > 180).mean() * 100
print(f"Recency > 180 days: {recency_pct:.1f}%")

# Month-1 retention (safe version)
if retention.shape[1] > 1:
    month1_retention = retention.iloc[:, 1].mean() * 100
    print(f"Month-1 retention: {month1_retention:.1f}%")
else:
    month1_retention = 0
    print("No customers had a repeat purchase in a later calendar month.")

# Top 10 sellers % of revenue
top10_sellers_query = """
SELECT SUM(total_item_value) AS top10_revenue
FROM orders_master
WHERE seller_id IN (
    SELECT seller_id FROM orders_master
    GROUP BY seller_id
    ORDER BY SUM(total_item_value) DESC
    LIMIT 10
);
"""
top10_revenue = pd.read_sql(top10_sellers_query, conn)['top10_revenue'][0]
total_revenue = df['total_item_value'].sum()
top10_pct = (top10_revenue / total_revenue) * 100
print(f"Top 10 sellers % of total revenue: {top10_pct:.1f}%")

Recency > 180 days: 60.5%
No customers had a repeat purchase in a later calendar month.
Top 10 sellers % of total revenue: 12.9%


## Key SQL Findings

- Monthly revenue grew steadily using the running total window function, confirming platform growth from Phase 2's chart with exact cumulative figures.
- 60.5% of customers have a recency of over 180 days, meaning most customers haven't purchased in over 6 months — a strong signal of low retention that reinforces the 3.1% repeat-purchase rate found in EDA.
- Cohort retention analysis shows essentially no customers return to purchase in a later calendar month — the retention table contains only period 0, meaning any repeat purchases happen within the same month as a customer's first order, not afterward. This points to retention/re-engagement as a significant, largely untapped opportunity.
- The top 10 sellers account for 12.9% of total platform revenue — worth flagging as a potential seller-concentration risk.